In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms # 데이터셋을 torchvision에서 가져옴
from torch.utils.data import DataLoader
from torchvision import transforms # 데이터 증강을 위해 추가

# 시드 고정 플래그
global fixSeed
fixSeed = False

In [ ]:
# 가중치 시드 고정을 하기 위해 실행합니다.
# 가중치 시드를 고정할 의도가 없는 경우 이 코드는 실행하지 않습니다.
import random
import numpy as np

# 시드를 고정하여 난수 고정
def seed_everything(seed=42):
    global fixSeed

    # 모델의 가중치 초기화, 드롭아웃 결과 고정
    random.seed(seed) # 파이썬 기본 난수 고정
    np.random.seed(seed) # NumPy 난수 고정
    torch.manual_seed(seed) # PyTorch CPU 연산 고정
    torch.cuda.manual_seed(seed) # 현재 GPU 연산 고정
    torch.cuda.manual_seed_all(seed) # 모든 GPU 연산 고정

    # 같은 입력에 항상 같은 출력
    torch.backends.cudnn.deterministic = True

    # cuDNN이 가장 빠른 알고리즘을 자동 선택하지 않게 하여, 실행 결과 변동 방지
    torch.backends.cudnn.benchmark = False

    # 시드 고정 플래그 활성화
    fixSeed = True

seed_everything(42)

In [ ]:
# 파라미터 세팅
batch_size = 32 # 배치 사이즈
learning_rate = 0.001 # 학습률
epoch = 15 # 에폭

# GPU 세팅
cuda_available = torch.cuda.is_available()

# cuda(NVIDIA GPU)를 사용 가능할 경우, cuda를 사용하여 병렬 처리
device = torch.device("cuda" if cuda_available else "cpu")

# 현재 불러온 device를 출력
print('Current device is', device)

Current device is cuda


In [ ]:
# 데이터셋 불러오기
train_data = datasets.MNIST(root='./data/train/', train=True, transform=transforms.ToTensor(), download=True)
test_data = datasets.MNIST(root='./data/test/', train=False, transform=transforms.ToTensor(), download=True)
print()

# 데이터셋 섞는 순서 고정
if fixSeed: # 시드를 고정한 경우, 마찬가지로 데이터셋을 고정으로 섞음
  g = torch.Generator()
  g.manual_seed(42)
  print("데이터셋 셔플 시드를 고정합니다.")
else: # 시드를 고정하지 않은 경우 데이터셋을 임의로 섞음
  print("데이터셋을 랜덤으로 셔플합니다.")

# 데이터를 섞어, 학습에 편향이 발생하지 않도록 함
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, generator=g if fixSeed else None)
test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=True, generator=g if fixSeed else None)

100%|██████████| 9.91M/9.91M [00:00<00:00, 65.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.70MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 17.4MB/s]
100%|██████████| 9.91M/9.91M [00:00<00:00, 59.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 3.70MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 17.4MB/s]

데이터셋 셔플 시드를 고정합니다.


In [ ]:
# 딥러닝 모델 설정
class Convolution_Neural_Networks(nn.Module): #nn.Module로부터 상속
    def __init__(self):
        super(Convolution_Neural_Networks, self).__init__() # CNN 클래스의 부모 클래스 생성자 호출

        # [conv1] 입력 -> 출력 예시: (1, 28, 28) -> (32, 28, 28)
        self.conv1 = nn.Conv2d(in_channels = 1, # 입력 채널 수: 28x28 흑백 이미지 1개
                               out_channels = 32, # 출력 채널 수: 32개의 필터
                               kernel_size=3, # 필터 크기: 3x3
                               stride=1, # 필터를 1칸씩 이동. 해상도 유지
                               padding='same' # 입력과 출력 크기를 동일하게 유지(28x28)
                               )

        # (개선) BatchNorm2d: 출력 채널 수인 32를 인자로 전달
        self.bn1 = nn.BatchNorm2d(32)

        # [conv2] 입력 -> 출력 예시: (32, 28, 28) -> (64, 28, 28)
        self.conv2 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size=3, stride=1, padding='same')

        # (개선) BatchNorm2d: 출력 채널 수인 64를 인자로 전달
        self.bn2 = nn.BatchNorm2d(64)

        self.dropout = nn.Dropout2d(0.25) # 학습 중 25%의 채널을 무작위로 0으로 만듦 (시드 고정시, 항상 고정된 값이 0이 됨)
        self.relu = nn.ReLU() # ReLU를 활성화 함수로 사용
        self.max_pooling = nn.MaxPool2d(kernel_size=2) # 2x2 영역에서 가장 큰 값 하나만 선택 (채널 수 유지, 해상도 절반으로 감소)

        # Fully Connected
        # [fc1] 입력 -> 출력: 3136 -> 1000
        self.fc1 = nn.Linear(7*7*64, 1000)

        # (개선) Linear 계층에서는 BatchNorm1d 사용
        self.bn3 = nn.BatchNorm1d(1000)

        # [fc2] 입력 -> 출력: 1000 -> 10 (최종 정답 레이어는 10가지)
        self.fc2 = nn.Linear(1000, 10)

    # 순전파 연산 : PyTorch는 기본적으로 역전파 자동 계산하므로 구현 필요 x
    def forward(self, x):
        # 1번째 conv : input(1, 28, 28) -> conv1(32, 28, 28) -> bn1 -> ReLU(32, 28, 28) -> max_pooling(32, 14, 14)
        x = self.conv1(x)
        x = self.bn1(x) # 배치 정규화 추가
        x = self.relu(x)
        x = self.max_pooling(x)

        # 2번째 conv : (32, 14, 14) -> conv1(64, 14, 14) -> bn2 -> ReLU(64, 14, 14) -> max_pooling(64, 7, 7)
        x = self.conv2(x)
        x = self.bn2(x) # 배치 정규화 추가
        x = self.relu(x)
        x = self.max_pooling(x)

        x = self.dropout(x) # 특징 추출 이후, 특정 채널 과의존 방지 dropout
        x = torch.flatten(x, 1) # Fully Connected Layer에 넣기 위해 (64, 7, 7)을 1차원으로 변환

        # 1번째 fc : 3136 -> 1000
        x = self.fc1(x)
        x = self.bn3(x) # 배치 정규화 추가
        x = self.relu(x)

        # 2번째 fc : 1000 -> output(10)
        x = self.fc2(x)

        output = F.log_softmax(x, dim=1) # 각 클래스를 로그 확률로 바꿔, 분류 손실 계산에 쓰기 위해 변환
        return output

In [ ]:
# 모델에 cpu/gpu 적용
model = Convolution_Neural_Networks().to(device) # CNN 모델 생성

# 실제 학습이 어디서 작동하는지 확인
print(f"Current device is {device}.")

# 가중치를 어떻게, 어떤 규칙으로 업데이트할지에 대한 모든 정보를 담은 객체
optimizer = optim.Adam(model.parameters(), lr=learning_rate) # Adam 옵티마이저 사용

# 손실 함수를 계산하는 함수 객체
criterion = nn.CrossEntropyLoss() # crossEntropy 사용

Current device is cuda.


In [ ]:
# 학습
i = 1
for epoch in range(epoch): # 에폭
  for data, target in train_loader: # 배치 반복
    data = data.to(device)
    target = target.to(device)
    optimizer.zero_grad() # gradient 초기화
    output = model(data) # forward 자동 호출
    loss = criterion(output, target) # 손실 계산
    loss.backward() # 역전파 연산
    optimizer.step() # 가중치 업데이트

    if i % 1000 == 0:
      print("Train Step: {}\tLoss : {:.3f}".format(i, loss.item()))

    i += 1

Train Step: 1000	Loss : 0.060
Train Step: 2000	Loss : 0.034
Train Step: 3000	Loss : 0.005
Train Step: 4000	Loss : 0.002
Train Step: 5000	Loss : 0.085
Train Step: 6000	Loss : 0.005
Train Step: 7000	Loss : 0.019
Train Step: 8000	Loss : 0.146
Train Step: 9000	Loss : 0.003
Train Step: 10000	Loss : 0.000
Train Step: 11000	Loss : 0.006
Train Step: 12000	Loss : 0.001
Train Step: 13000	Loss : 0.002
Train Step: 14000	Loss : 0.020
Train Step: 15000	Loss : 0.002
Train Step: 16000	Loss : 0.005
Train Step: 17000	Loss : 0.000
Train Step: 18000	Loss : 0.003
Train Step: 19000	Loss : 0.000
Train Step: 20000	Loss : 0.018
Train Step: 21000	Loss : 0.000
Train Step: 22000	Loss : 0.013
Train Step: 23000	Loss : 0.000
Train Step: 24000	Loss : 0.005
Train Step: 25000	Loss : 0.000
Train Step: 26000	Loss : 0.000
Train Step: 27000	Loss : 0.002
Train Step: 28000	Loss : 0.000


In [ ]:
# 평가
model.eval() # 평가 모드: dropout 비활성화, 배치 정규화를 고정값으로 사용
correct = 0
for data, target in test_loader:
  data = data.to(device)
  target = target.to(device)
  output = model(data) # 클래스별 점수 출력
  prediction = output.data.max(1)[1] # 각 샘플에서 가장 큰 값의 인덱스
  correct += prediction.eq(target.data).sum()
print('Test set Accuracy : {:.2f}%'.format(100. * correct / len(test_loader.dataset)))

Test set Accuracy : 99.30%


In [ ]:
# 모델 저장
PATH = '/content/drive/MyDrive/CNN_improved1.pt'

torch.save(model.state_dict(), PATH)